In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import xarray as xr
import seaborn as sns
sns.set_theme("notebook",style="dark")

In [ ]:
import os
from pathlib import Path

from qgsw.logging.core import getLogger, setup_root_logger
from qgsw.specs import defaults


torch.backends.cudnn.deterministic = True

ROOT_PATH = Path(os.getcwd()).parent


specs=defaults.get()

logger=getLogger(__name__)
setup_root_logger(1)

In [ ]:
from collections.abc import Callable
from typing import Any

from matplotlib import pyplot as plt

from qgsw import plots
from qgsw.scripts.output import build_output_folder


class LossReader:

    _label: str
    color: str = "black"
    linestyle: str = "solid"
    cycle = -1

    show = True

    @property
    def plot_kwargs(self) -> dict[str, Any]:
        """Plotting kwargs."""
        return {
            "color": self.color,
            "linestyle": self.linestyle,
            "label": self.label,
        }

    @property
    def label(self) -> str:
        try:
            return self._label
        except AttributeError:
            self._label = self.params["model_type"]
            return self._label

    def __init__(self, output_folder:str|Path) -> None:
        if not Path(output_folder).exists():
            msg = f"Folder {output_folder} does not exist."
            logger.warning(msg)
            self.show=False
            return
        output = build_output_folder(output_folder)
        if not output.params.file.exists() or not output.scores.file.exists():
            self.show=False
            return
        self.params = output.params.load()
        self.scores = output.scores.load()
        self.data_dt = self.scores.pop("data_dt")

    def plot(self, loss_name:str, cycles:int|list[int]|None=None,*,ax:plt.Axes|None=None) -> None:
        if not self.show:return
        if cycles is None:
            cycles = list(range(len(self.scores[list(self.scores)[0]])))
        elif isinstance(cycles,int):
            cycles = [cycles]
        time_offset = 0
        for i, c in enumerate(cycles):
            steps = np.arange(len(self.scores[loss_name][c]))
            times = self.data_dt * steps / 3600 / 24 + time_offset
            time_offset = times[-1] + self.data_dt / 3600 / 24
            loss = self.scores[loss_name][c].cpu().numpy()
            kwargs = self.plot_kwargs.copy()
            if i != 0:
                kwargs.pop("label")
            ax.plot(times, loss, **kwargs)
        ax.set_xlabel("Time [day]")

def check_indices(*loss_readers: LossReader)-> None:
    readers = [loss_reader for loss_reader in loss_readers if loss_reader.show]
    if not readers:return
    indices = [loss_reader.params["indices"] for loss_reader in readers]
    if not all(e==indices[0] for e in indices):
        msg = "Losses evaluated on different areas."
        raise ValueError(msg)

def check_season(*loss_readers: LossReader)-> None:
    readers = [loss_reader for loss_reader in loss_readers if loss_reader.show]
    if not readers:return
    seasons = [loss_reader.params["season"] for loss_reader in readers]
    if not all(e==seasons[0] for e in seasons):
        msg = "Losses evaluated on different seasons."
        raise ValueError(msg)

def plot(*loss_readers:LossReader,with_grad:bool=True, with_psi2:bool=True,cycles: int|list[int]|None=None,check_fn:Callable[[LossReader],None]|None=None) -> np.ndarray:

    if check_fn:check_fn(*loss_readers)

    readers = [loss_reader for loss_reader in loss_readers if loss_reader.show]    

    show_grad = with_grad and  any(["uv_rmse" in loss_reader.scores.keys() for loss_reader in readers])
    show_psi2 = with_psi2 and  any(["psi2_rmse" in loss_reader.scores.keys() for loss_reader in readers])

    fig,axs = plots.subplots(1+show_grad+show_psi2,1,figsize=(22,(5)*(1+show_grad+show_psi2)))
    plots.set_rowtitles(["RMSE"]+["UV RMSE"]*show_grad + ["Psi2 RMSE"]*show_psi2,axs)
    
    for loss_reader in readers:
        loss_reader.plot("rmse",cycles,ax=axs[0,0])
    if show_grad:
        for loss_reader in readers:
            loss_reader.plot("uv_rmse",cycles,ax=axs[1,0])
    if show_psi2:
        for loss_reader in readers:
            loss_reader.plot("psi2_rmse",cycles,ax=axs[1+show_grad,0])

    plots.clamp_ylims(0,1,axs)
    axs[0,0].legend()

    return fig, axs

## [ 32, 96 ] x [ 256, 384 ]

In [ ]:
rg0 = LossReader("../output/g5k/assimilation/QG3L_RG0-20261116_101610-b65015d9")
rgf=LossReader("../output/g5k/assimilation/QG3L_RGF-20261113_214548-338a3939")
rgf.color = "brown"
rgfsi=LossReader("../output/g5k/assimilation/QG3L_RGFSI-20261114_055455-6c48cb08")
rgfsi.color="green"
rgsi=LossReader("../output/g5k/assimilation/QG3L_RGSI-20261113_172507-aa49b5dd")
rgsi.color="navy"

loss_readers_z1 = [rg0,rgsi,rgf,rgfsi]

fig, axs  =plot(*loss_readers_z1,check_fn=check_indices)

plt.suptitle("[ 32, 96 ] x [ 256, 384 ]")

## [ 112, 176 ] x [ 64, 192 ]

In [ ]:
rg0=LossReader("../output/g5k/assimilation/QG3L_RG0-20261116_104632-66a8cc9c")
rgf=LossReader("../output/g5k/assimilation/QG3L_RGF-20261113_222934-3053b33f")
rgf.color = "brown"
rgfsi=LossReader("../output/g5k/assimilation/QG3L_RGFSI-20261114_074456-aee5d3ef")
rgfsi.color="green"
rgsi=LossReader("../output/g5k/assimilation/QG3L_RGSI-20261113_172507-e2e0eed4")
rgsi.color="navy"

loss_readers_z2 = [rg0,rgsi,rgf,rgfsi]

fig, axs = plot(*loss_readers_z2,check_fn=check_indices)


plt.suptitle("[ 112, 176 ] x [ 64, 192 ]")

### Summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import Size, Divider
import matplotlib.ticker as ticker
from math import sqrt

gratio = (1+sqrt(5))/2


fig = plt.figure(dpi=150)

# Define fixed sizes in inches
pad_left  = Size.Fixed(0.8)
ax_width  = Size.Fixed(5.0)
gap       = Size.Fixed(0.1)
pad_right = Size.Fixed(0.3)

row_height = Size.Fixed(5 / gratio)
row_gap    = Size.Fixed(0.4)   # vertical gap between the two rows

h = [pad_left, ax_width, gap, ax_width, pad_right]
v = [
    Size.Fixed(0.6),   # bottom margin
    row_height,        # bottom row (row 1)
    row_gap,           # gap between rows
    row_height,        # top row (row 2)
    Size.Fixed(0.4),   # top margin
]

divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)

# Bottom row  (ny=1)
ax1_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=1))
ax2_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=1),
                       sharey=ax1_bot)

# Top row  (ny=3)
ax1_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=3),
                        sharex=ax1_bot)
ax2_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=3),
                       sharey=ax1_top, sharex=ax2_bot)

times = np.arange(250) * 7200

# ── plotting helper ───────────────────────────────────────────────────────────
# def plot_on(ax_summer, ax_winter, losses_dict):

for reader in loss_readers_z1:
    if not reader.show:
        continue
    t = times / 3600 / 24

    ls = torch.stack(reader.scores["rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_top.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax1_top.plot(t, m, **reader.plot_kwargs)

    ls = torch.stack(reader.scores["uv_rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_bot.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax1_bot.plot(t, m, **reader.plot_kwargs)

for reader in loss_readers_z2:
    if not reader.show:
        continue
    t = times / 3600 / 24

    ls = torch.stack(reader.scores["rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_top.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax2_top.plot(t, m, **reader.plot_kwargs)
    
    ls = torch.stack(reader.scores["uv_rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_bot.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax2_bot.plot(t, m, **reader.plot_kwargs)

# ── shared spine / tick style ─────────────────────────────────────────────────
def style_ax(ax, hide_yticks=False, hide_xticks=False):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_bounds(0, 20)
    ax.spines["left"].set_bounds(0, 1)
    ax.spines["bottom"].set_position(("outward", 5))
    ax.spines["left"].set_position(("outward", 5))
    ax.tick_params(axis='both', labelsize="large")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))
    if hide_yticks:
        ax.tick_params(labelleft=False)
    if hide_xticks:
        ax.tick_params(labelbottom=False)
        ax.spines["bottom"].set_visible(False)

style_ax(ax1_bot)
style_ax(ax2_bot, hide_yticks=True)
style_ax(ax1_top, hide_xticks=True)          # no x-axis on top row
style_ax(ax2_top, hide_yticks=True, hide_xticks=True)

# ── labels & titles ───────────────────────────────────────────────────────────
ax1_top.set_title("Z1", fontsize="x-large")
ax2_top.set_title("Z2",  fontsize="x-large")

ax1_bot.set_xlabel("Time [days]", fontsize="large")
ax2_bot.set_xlabel("Time [days]", fontsize="large")

ax1_top.set_ylabel("Stream function nRMSE", fontsize="large")
ax1_bot.set_ylabel("Velocity nRMSE", fontsize="large")

ax1_top.legend(fontsize="large")
plots.clamp_ylims(0, 1, ax1_top)
plots.clamp_ylims(0, 1, ax1_bot)
plots.set_ylims(0, None, ax1_bot)
plots.set_ylims(0, None, ax1_top)

# fig.savefig("../output/images/enatl60_all_qg3l_rmses.svg", bbox_inches="tight")

# eNATL60

## Summer

In [ ]:
out_folder = Path("../output/g5k/assimilation")


rg0 = LossReader(out_folder / "eNATL60_RG0-20261123_164432-5b1331d6")
rgf=LossReader(out_folder / "eNATL60_RGF-20261123_110127-a3d7b069")
rgf.color = "brown"
rgf_vd = LossReader(out_folder / "eNATL60_RGF-20261123_130022-202c6215")
rgf_vd.color="brown"
rgf_vd.linestyle="dashed"
rgfsi=LossReader(out_folder / "eNATL60_RGFSI-20261123_150718-438d6a48")
rgfsi.color="green"
rgsi=LossReader(out_folder / "eNATL60_RGSI-20261123_095445-f8063ce7")
rgsi.color="navy"
rgsi_vd=LossReader(out_folder / "eNATL60_RGSI-20261123_104532-c8d0d5a5")
rgsi_vd.color="navy"
rgsi_vd.linestyle="dashed"

loss_readers_summer = [rg0,rgsi,rgsi_vd,rgf,rgf_vd,rgfsi]

fig, axs  =plot(*loss_readers_summer,check_fn=check_season)

plt.suptitle("Summer")

## Winter

In [ ]:
out_folder = Path("../output/g5k/assimilation")

rg0 = LossReader(out_folder/"eNATL60_RG0-20261123_164601-e733fbfb")
rgf=LossReader(out_folder/"eNATL60_RGF-20261123_124210-20dc18fa")
rgf.color = "brown"
rgf_vd = LossReader(out_folder/"eNATL60_RGF-20261123_145809-08d61158")
rgf_vd.color="brown"
rgf_vd.linestyle="dashed"
rgfsi=LossReader(out_folder/"eNATL60_RGFSI-20261123_161725-dfd7b16b")
rgfsi.color="green"
rgsi=LossReader(out_folder/"eNATL60_RGSI-20261123_095445-20453217")
rgsi.color="navy"
rgsi_vd=LossReader(out_folder/"eNATL60_RGSI-20261123_104626-cddbe4bc")
rgsi_vd.color="navy"
rgsi_vd.linestyle="dashed"

loss_readers_winter = [rg0,rgsi,rgsi_vd,rgf,rgf_vd,rgfsi]

fig, axs  =plot(*loss_readers_winter,check_fn=check_season)

plt.suptitle("Winter")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import Size, Divider
import matplotlib.ticker as ticker
from math import sqrt

gratio = (1+sqrt(5))/2


fig = plt.figure(dpi=150)

# Define fixed sizes in inches
pad_left  = Size.Fixed(0.8)
ax_width  = Size.Fixed(5.0)
gap       = Size.Fixed(0.1)
pad_right = Size.Fixed(0.3)

row_height = Size.Fixed(5 / gratio)
row_gap    = Size.Fixed(0.4)   # vertical gap between the two rows

h = [pad_left, ax_width, gap, ax_width, pad_right]
v = [
    Size.Fixed(0.6),   # bottom margin
    row_height,        # bottom row (row 1)
    row_gap,           # gap between rows
    row_height,        # top row (row 2)
    Size.Fixed(0.4),   # top margin
]

divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)

# Bottom row  (ny=1)
ax1_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=1))
ax2_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=1),
                       sharey=ax1_bot)

# Top row  (ny=3)
ax1_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=3),
                        sharex=ax1_bot)
ax2_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=3),
                       sharey=ax1_top, sharex=ax2_bot)

times = np.arange(240) * 7200

# ── plotting helper ───────────────────────────────────────────────────────────
# def plot_on(ax_summer, ax_winter, losses_dict):

for reader in loss_readers_summer:
    if not reader.show:
        continue
    t = times / 3600 / 24

    ls = torch.stack(reader.scores["rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_top.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax1_top.plot(t, m, **reader.plot_kwargs)

    ls = torch.stack(reader.scores["uv_rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_bot.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax1_bot.plot(t, m, **reader.plot_kwargs)

for reader in loss_readers_winter:
    if not reader.show:
        continue
    t = times / 3600 / 24

    ls = torch.stack(reader.scores["rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_top.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax2_top.plot(t, m, **reader.plot_kwargs)
    
    ls = torch.stack(reader.scores["uv_rmse"])
    not_nan = ~torch.isnan(ls).any(dim=1)
    ms   = torch.mean(ls[not_nan], dim=0)
    stds = torch.std(ls[not_nan],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_bot.fill_between(t, m - s, m + s, alpha=0.2, color=reader.color)
    ax2_bot.plot(t, m, **reader.plot_kwargs)

# ── shared spine / tick style ─────────────────────────────────────────────────
def style_ax(ax, hide_yticks=False, hide_xticks=False):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_bounds(0, 20)
    ax.spines["left"].set_bounds(0, 1)
    ax.spines["bottom"].set_position(("outward", 5))
    ax.spines["left"].set_position(("outward", 5))
    ax.tick_params(axis='both', labelsize="large")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))
    if hide_yticks:
        ax.tick_params(labelleft=False)
    if hide_xticks:
        ax.tick_params(labelbottom=False)
        ax.spines["bottom"].set_visible(False)

style_ax(ax1_bot)
style_ax(ax2_bot, hide_yticks=True)
style_ax(ax1_top, hide_xticks=True)          # no x-axis on top row
style_ax(ax2_top, hide_yticks=True, hide_xticks=True)

# ── labels & titles ───────────────────────────────────────────────────────────
ax1_top.set_title("Z1", fontsize="x-large")
ax2_top.set_title("Z2",  fontsize="x-large")

ax1_bot.set_xlabel("Time [days]", fontsize="large")
ax2_bot.set_xlabel("Time [days]", fontsize="large")

ax1_top.set_ylabel("Stream function nRMSE", fontsize="large")
ax1_bot.set_ylabel("Velocity nRMSE", fontsize="large")

ax1_top.legend(fontsize="large")
plots.clamp_ylims(0, 1, ax1_top)
plots.clamp_ylims(0, 1, ax1_bot)
plots.set_ylims(0, None, ax1_bot)
plots.set_ylims(0, None, ax1_top)

# fig.savefig("../output/images/enatl60_all_qg3l_rmses.svg", bbox_inches="tight")